In [1]:
import polars as pl
import numpy as np
import os

# Load S350 dataset
# CSV is without header, so we specify the column names
df = pl.read_csv(
    "../data/S350_fixed.csv",
    has_header=False,
    new_columns=["name", "mutation", "from", "relative_position_offset", "ddG"],
)

df

name,mutation,from,relative_position_offset,ddG
str,str,str,i64,f64
"""1AJ3""","""H10A""","""A""",-9,-0.5
"""1AJ3""","""I23A""","""A""",-9,3.6
"""1AJ3""","""E25A""","""A""",-9,-0.1
"""1AJ3""","""K26A""","""A""",-9,0.0
"""1AJ3""","""V30A""","""A""",-9,0.2
…,…,…,…,…
"""5DFR""","""G121H""","""A""",0,0.56
"""5DFR""","""I155T""","""A""",0,2.53
"""5PTI""","""A16T""","""A""",0,1.7


In [2]:
import polars as pl
import requests
import json
import os
import time
import re
from Bio import SeqIO
from io import StringIO

# Create cache directory
if not os.path.exists(".cache"):
    os.makedirs(".cache")

# --- CONFIGURATION ---
CACHE_FILE = ".cache/pdb_chains_cache.json"
MAPPING_OUTPUT_FILE = ".cache/final_id_to_sequence_map.json"

# 1. DATA PREPARATION
df = df.with_columns(pl.col("name").str.strip_chars('" ').alias("clean_id"))
unique_pdb_roots = df["clean_id"].str.slice(0, 4).str.to_uppercase().unique().to_list()


# 2. FUNCTION TO PARSE FASTA
def parse_fasta_to_dict(fasta_string):
    if not fasta_string: return {}
    chains = {}
    try:
        file = StringIO(fasta_string)
        for record in SeqIO.parse(file, "fasta"):
            header = record.description
            seq = str(record.seq)
            # Extract chain ID from header
            match = re.search(r"Chain[s]?\s+([A-Za-z0-9, ]+)", header)
            if match:
                chain_id = match.group(1).split(",")[0].strip()
            else:
                chain_id = record.id
            chains[chain_id] = seq
    except Exception as e:
        print(f"Error parsing FASTA: {e}")
        return {}
    return chains


# 3. LOAD / DOWNLOAD CACHE
chains_cache = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, "r") as f:
            chains_cache = json.load(f)
    except:
        chains_cache = {}

ids_to_download = [pid for pid in unique_pdb_roots if pid not in chains_cache]

if ids_to_download:
    print(f"Downloading {len(ids_to_download)} PDB records...")
    for i, pdb_id in enumerate(ids_to_download):
        try:
            url = f"https://www.rcsb.org/fasta/entry/{pdb_id}"
            response = requests.get(url, timeout=10)
            chains_cache[pdb_id] = parse_fasta_to_dict(response.text) if response.status_code == 200 else {}
            if i % 10 == 0: time.sleep(0.1)
        except:
            chains_cache[pdb_id] = {}
    with open(CACHE_FILE, "w") as f:
        json.dump(chains_cache, f)

# 4. VALIDATION AND MUTATION GENERATION
print("\nStarting validation and sequence generation...")
results = []
rows = df.select(["clean_id", "mutation", "relative_position_offset"]).to_dicts()

valid_count = 0
not_downloaded_count = 0

for row in rows:
    pdb_id = row["clean_id"]
    pdb_root = pdb_id[:4].upper()
    mutation = row["mutation"].replace('"', '').strip()
    offset = row["relative_position_offset"]

    chains_dict = chains_cache.get(pdb_root, {})

    # Parse mutation (e.g., A123V -> WT: A, Pos: 123, MT: V)
    match = re.match(r'^([A-Z])(\d+)([A-Z])$', mutation, re.IGNORECASE)

    if not match or not chains_dict:
        results.append({
            "wt_sequence": None,
            "mut_sequence": None,
            "match_info": "Fail (Data or Format)",
            "is_valid": False
        })
        if not chains_dict: not_downloaded_count += 1
        continue

    wt_expected = match.group(1).upper()
    pos_num = int(match.group(2))
    mt_residue = match.group(3).upper()

    found_wt_seq = None
    found_mut_seq = None
    worked_idx = -1
    match_info = "WT Mismatch"
    is_valid = False

    # Strategies to find the correct residue position
    strategies = [
        ("Offset", lambda p, o: p + (o if o is not None else 0) - 1),
        ("Direct", lambda p, o: p - 1),
        ("Raw", lambda p, o: p),
        ("Fuzzy-1", lambda p, o: p - 2),
    ]

    sorted_chains = sorted(chains_dict.items(), key=lambda x: len(x[1]), reverse=True)

    for chain_name, seq in sorted_chains:
        for strat_name, func in strategies:
            try:
                idx = func(pos_num, offset)
                if 0 <= idx < len(seq) and seq[idx].upper() == wt_expected:
                    worked_idx = idx
                    found_wt_seq = seq
                    # Create mutated sequence
                    seq_list = list(seq)
                    seq_list[idx] = mt_residue
                    found_mut_seq = "".join(seq_list)

                    match_info = f"Chain:{chain_name}|{strat_name}"
                    is_valid = True
                    break
            except:
                pass
        if is_valid: break

    # If invalid, take the longest chain as fallback for WT
    fallback_wt = sorted_chains[0][1] if not is_valid and sorted_chains else None

    results.append({
        "wt_sequence": found_wt_seq if is_valid else fallback_wt,
        "mut_sequence": found_mut_seq,  # Will be None if is_valid is False
        "match_info": match_info,
        "is_valid": is_valid
    })

    if is_valid: valid_count += 1

# 5. RESULTS
result_df = pl.DataFrame(results)
df_final = pl.concat([df, result_df], how="horizontal")

print(f"Done. ✅ Validated: {valid_count} / {len(df_final)}")

df_final


Starting validation and sequence generation...
Done. ✅ Validated: 322 / 350


name,mutation,from,relative_position_offset,ddG,clean_id,wt_sequence,mut_sequence,match_info,is_valid
str,str,str,i64,f64,str,str,str,str,bool
"""1AJ3""","""H10A""","""A""",-9,-0.5,"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLAQFFRDMDDEESWIKEKKLLV…","""Chain:A|Direct""",true
"""1AJ3""","""I23A""","""A""",-9,3.6,"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWAKEKKLLV…","""Chain:A|Direct""",true
"""1AJ3""","""E25A""","""A""",-9,-0.1,"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWIKAKKLLV…","""Chain:A|Direct""",true
"""1AJ3""","""K26A""","""A""",-9,0.0,"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWIKEAKLLV…","""Chain:A|Direct""",true
"""1AJ3""","""V30A""","""A""",-9,0.2,"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWIKEKKLLA…","""Chain:A|Direct""",true
…,…,…,…,…,…,…,…,…,…
"""5DFR""","""G121H""","""A""",0,0.56,"""5DFR""","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""Chain:A|Offset""",true
"""5DFR""","""I155T""","""A""",0,2.53,"""5DFR""","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""Chain:A|Offset""",true
"""5PTI""","""A16T""","""A""",0,1.7,"""5PTI""","""RPDFCLEPPYTGPCKARIIRYFYNAKAGLC…","""RPDFCLEPPYTGPCKTRIIRYFYNAKAGLC…","""Chain:A|Offset""",true


In [3]:
# Filter valid records and select final columns
df_final = df_final.filter(pl.col("is_valid")).select(["name", "wt_sequence", "mut_sequence", "mutation", "ddG", "match_info"])
df_final

name,wt_sequence,mut_sequence,mutation,ddG,match_info
str,str,str,str,f64,str
"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLAQFFRDMDDEESWIKEKKLLV…","""H10A""",-0.5,"""Chain:A|Direct"""
"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWAKEKKLLV…","""I23A""",3.6,"""Chain:A|Direct"""
"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWIKAKKLLV…","""E25A""",-0.1,"""Chain:A|Direct"""
"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWIKEAKLLV…","""K26A""",0.0,"""Chain:A|Direct"""
"""1AJ3""","""AKLNESHRLHQFFRDMDDEESWIKEKKLLV…","""AKLNESHRLHQFFRDMDDEESWIKEKKLLA…","""V30A""",0.2,"""Chain:A|Direct"""
…,…,…,…,…,…
"""5DFR""","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""G121H""",0.56,"""Chain:A|Offset"""
"""5DFR""","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""MISLIAALAVDRVIGMENAMPWNLPADLAW…","""I155T""",2.53,"""Chain:A|Offset"""
"""5PTI""","""RPDFCLEPPYTGPCKARIIRYFYNAKAGLC…","""RPDFCLEPPYTGPCKTRIIRYFYNAKAGLC…","""A16T""",1.7,"""Chain:A|Offset"""


In [4]:
# Normalize ddG using sigmoid function
k_neg = 0.230
k_pos = 0.576
A_neg = 1.0
A_pos = 1.0

sigmoid_expr = (
    pl.when(pl.col("ddG") >= 0)
    .then(A_pos * (2 / (1 + (-k_pos * pl.col("ddG")).exp()) - 1))
    .otherwise(-A_neg * (2 / (1 + (-k_neg * (-pl.col("ddG"))).exp()) - 1))
)

df_final = df_final.with_columns(
    sigmoid_expr.alias("target")
)

# Create datasets directory and save to Parquet
if not os.path.exists("datasets"):
    os.makedirs("datasets")

df_final.select(["wt_sequence", "mut_sequence", "mutation", "target", "match_info"]).drop_nulls(
    ["mut_sequence", "mutation"]).write_parquet("datasets/s350_dataset.parquet")
